In [8]:
import cv2
import torch
from ultralytics import YOLO

In [11]:
# Load the trained YOLO classification model
classify_model = YOLO("runs/classify/train/weights/best.pt")

# Load the input video
test_vod = "VODS/test.webm"
cap = cv2.VideoCapture(test_vod)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define output video writer
output_path = "useful.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for mp4 format
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = classify_model(rgb_frame)
    prediction = results[0].probs.top1
    print("Prediction:", prediction)  # Confirm prediction value

    if prediction == 1:
        out.write(frame)
        print("Frame written.")  # Log every time a frame is written

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Processed video saved as {output_path}")



0: 640x640 0 1.00, 1 0.00, 56.0ms
Speed: 24.0ms preprocess, 56.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Prediction: 0

0: 640x640 0 1.00, 1 0.00, 53.0ms
Speed: 22.0ms preprocess, 53.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Prediction: 0

0: 640x640 0 1.00, 1 0.00, 55.0ms
Speed: 22.0ms preprocess, 55.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Prediction: 0

0: 640x640 0 1.00, 1 0.00, 56.0ms
Speed: 28.0ms preprocess, 56.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Prediction: 0

0: 640x640 0 1.00, 1 0.00, 49.0ms
Speed: 23.0ms preprocess, 49.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Prediction: 0

0: 640x640 0 1.00, 1 0.00, 50.0ms
Speed: 26.0ms preprocess, 50.0ms inference, 0.0ms postprocess per image at shape (1, 3, 640, 640)
Prediction: 0

0: 640x640 0 1.00, 1 0.00, 60.0ms
Speed: 33.0ms preprocess, 60.0ms inference, 0.0ms postprocess per image at shape (1

KeyboardInterrupt: 

In [10]:
print("FPS:", fps)
print("Width:", width)
print("Height:", height)


FPS: 29
Width: 1280
Height: 720


In [ ]:
def skip_frame(result):
    if result == 0:
        current_frame = current_frame + 10*fps #skip 10 seconds worth of frames

In [6]:
import cv2
import numpy as np
import os

def test_video_capture(video_path):
    """Test if the input video can be opened and a frame can be read."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise AssertionError("Error: Video capture is not opened.")
    
    ret, frame = cap.read()
    if not ret or frame is None:
        raise AssertionError("Error: Could not read a frame from video capture.")
    
    print("Video capture test passed.")
    cap.release()

def test_video_writer(output_path, width, height, fps):
    """Test if the VideoWriter is configured properly by writing a dummy frame."""
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    if not writer.isOpened():
        raise AssertionError("Error: Video writer is not opened.")
    
    # Create a dummy frame (white image)
    dummy_frame = 255 * np.ones((height, width, 3), dtype=np.uint8)
    writer.write(dummy_frame)
    writer.release()
    
    # Check if file exists and has a reasonable size
    if not os.path.exists(output_path) or os.path.getsize(output_path) < 1000:
        raise AssertionError("Error: Output file may be corrupted or not written properly.")
    
    print("Video writer test passed.")

def test_processing_logic(video_path, output_path):
    """
    Test the processing loop by simulating classification.
    For testing purposes, assume every frame is classified as 'useful' (i.e. prediction == 1).
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise AssertionError("Error: Video capture is not opened.")

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    if not writer.isOpened():
        raise AssertionError("Error: Video writer is not opened.")
    
    frame_count = 0
    written_count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1
        
        # Simulate the classification process:
        # For testing, we assume every frame is "useful"
        prediction = 1  
        if prediction == 1:
            writer.write(frame)
            written_count += 1
    
    cap.release()
    writer.release()
    
    if written_count == 0:
        raise AssertionError("Error: No frames were written to the output.")
    
    print(f"Processing logic test passed. Processed {frame_count} frames, wrote {written_count} frames.")

if __name__ == "__main__":
    # Replace these paths with your actual test video and desired output path
    test_video_path = "VODS/test.webm"  
    test_output_path = "test_useful_frames.mp4"
    
    # Run video capture test
    test_video_capture(test_video_path)
    
    # Retrieve video properties for writer test
    cap = cv2.VideoCapture(test_video_path)
    if not cap.isOpened():
        raise AssertionError("Error: Could not open test video to get properties.")
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    
    # Run video writer test
    test_video_writer(test_output_path, width, height, fps)
    
    # Run processing logic test (simulate frame selection)
    test_processing_logic(test_video_path, test_output_path)


Video capture test passed.
Video writer test passed.
Processing logic test passed. Processed 4635 frames, wrote 4635 frames.
